# RQ VII: Goodreads author-gender representation


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reported_runs/analysis").is_dir())
sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None
from reported_runs.analysis.presentation import load_results, NotebookReport, temporal_table, effect_table, rank_summary, unique_trajectories, label_conditions
from reported_runs.analysis.plotting import trajectory_grid, effect_grid, rank_grid, association_grid
from reported_runs.analysis.comparisons import summarise_iterations
%matplotlib inline

results = load_results()
report = NotebookReport(results, 'rq7')
scales = report.scales
trajectory = label_conditions(results["trajectory"])
sources = results["sources"]
summaries = results["summaries"]
effect_summaries = results["effect_summaries"]
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 60)

# RQ VII — Author-gender representation and mitigation


In [ ]:
from reported_runs.analysis.sections_rq6_7 import (GENDER_ALL, GENDER_BINARY, GENDER_GAPS, COVERAGE, final_goodreads_conditions, transfer_effects, reference_table, gender_mitigation_effects)
rq_sources = sources.loc[sources.dataset.eq('goodreads')]
report.availability(rq_sources)
goodreads = final_goodreads_conditions(trajectory)

## Labelled catalogue and historical context


In [ ]:
references = results['references'].loc[results['references'].dataset.eq('goodreads')]
context = reference_table(references)
report.table(context, 'rq7_gender_reference_composition', "", columns=['seed', 'population_seed', 'catalogue_size', 'catalogue_female_share', 'initial_training_female_share', 'catalogue_unknown_share', 'catalogue_ambiguous_share'])

## Women-authored representation among all items


In [ ]:
figure = trajectory_grid(goodreads, GENDER_ALL, title='Women-authored representation among all items', scales=scales)
report.figure(figure, goodreads, 'rq7_gender_all_item_trajectories', "", recommended=False, why="")

In [ ]:
gr_summaries = summaries.loc[summaries.study.eq('goodreads_gender_a300_e804')]
table = temporal_table(gr_summaries, GENDER_ALL)
report.table(table, 'rq7_all_item_gender_summaries', "")

## Women-plus-men denominators


In [ ]:
figure = trajectory_grid(goodreads, GENDER_BINARY, title='Women-authored representation within women- and men-labelled items', scales=scales)
report.figure(figure, goodreads, 'rq7_gender_women_men_trajectories', "")

In [ ]:
table = temporal_table(gr_summaries, GENDER_BINARY)
report.table(table, 'rq7_women_men_gender_summaries', "")

## Selection relative to exposure


In [ ]:
figure = trajectory_grid(goodreads, GENDER_GAPS, title='Gender click-exposure gaps', scales=scales)
report.figure(figure, goodreads, 'rq7_gender_click_exposure_gaps', "")

In [ ]:
table = temporal_table(gr_summaries, GENDER_GAPS)
report.table(table, 'rq7_gender_click_exposure_gap_summaries', "")

## Matched feedback and response-model comparisons


In [ ]:
gender_effects = transfer_effects(effect_summaries)
gender_effects = gender_effects.loc[gender_effects.dataset.eq('goodreads')]
figure = effect_grid(gender_effects, GENDER_ALL, title='Matched gender-representation changes', scales=scales, facet='window', value='value')
report.figure(figure, gender_effects, 'rq7_matched_gender_feedback_effects', "")

In [ ]:
table = effect_table(gender_effects, 'female_click_share')
report.table(table, 'rq7_matched_women_authored_click_effects', "")

## Candidate availability before each canonical decision


In [ ]:
candidate_metrics = ['eligible_female_share', 'female_exposure_minus_candidate_share', 'weighted_female_exposure_minus_candidate_share']
figure = trajectory_grid(goodreads, candidate_metrics, title='Women-authored exposure relative to candidate availability', scales=scales)
report.figure(figure, goodreads, 'rq7_gender_candidate_availability', "")

In [ ]:
table = temporal_table(gr_summaries, candidate_metrics)
report.table(table, 'rq7_candidate_availability_summaries', "")

## Unknown and ambiguous label coverage


In [ ]:
figure = trajectory_grid(goodreads, COVERAGE[:3], title='Unknown-label exposure and click composition', scales=scales)
report.figure(figure, goodreads, 'rq7_unknown_label_coverage', "")

In [ ]:
figure = trajectory_grid(goodreads, COVERAGE[3:], title='Ambiguous-label exposure and click composition', scales=scales)
report.figure(figure, goodreads, 'rq7_ambiguous_label_coverage', "")

In [ ]:
table = temporal_table(gr_summaries, COVERAGE)
report.table(table, 'rq7_unresolved_label_coverage_summaries', "")

## Clicking behaviour


In [ ]:
figure = trajectory_grid(goodreads, ['click_rate', 'no_click_rate', 'valid_clicks'], title='Goodreads clicking context for gender representation', scales=scales)
report.figure(figure, goodreads, 'rq7_click_behaviour', "")

In [ ]:
table = temporal_table(gr_summaries, ['click_rate', 'no_click_rate', 'valid_clicks'])
report.table(table, 'rq7_click_behaviour_summaries', "")

## Computational gender mitigation


In [ ]:
mitigation = gender_mitigation_effects(effect_summaries)
mitigation_l2 = mitigation.loc[mitigation.stage.eq('L2_ml_model')]
figure = effect_grid(mitigation_l2, GENDER_ALL, title='L2 gender-mitigation effects', scales=scales, facet='window', value='value')
report.figure(figure, mitigation_l2, 'rq7_gender_mitigation_l2_effects', "", recommended=False, why="")

In [ ]:
mitigation_mean = mitigation.loc[mitigation.window.eq('mean_1_30')].assign(
    method=lambda frame: frame.intervention.map({'in_processing': 'In-processing', 'post_processing': 'Post-processing'}))
figure = effect_grid(mitigation_mean, ['female_click_share', 'weighted_female_exposure'], title='Mean gender-mitigation effects across L0, L1, and L2', scales=scales, category='method', facet='stage', value='value')
report.figure(figure, mitigation_mean, 'rq7_gender_mitigation_all_stages', "")

## How mitigation changes click-exposure gaps


In [ ]:
figure = effect_grid(mitigation_l2, ['female_click_exposure_gap', 'weighted_female_click_exposure_gap'], title='L2 gender-mitigation effects on click-exposure gaps', scales=scales, facet='window', value='value')
report.figure(figure, mitigation_l2, 'rq7_gender_mitigation_gap_effects', "")

In [ ]:
table = effect_table(mitigation, 'female_click_share')
report.table(table, 'rq7_gender_mitigation_click_effects', "")

In [ ]:
method_effects = gender_mitigation_effects(effect_summaries, include_method=True)
method_effects = method_effects.loc[method_effects.kind.eq('mitigation_method')]
method_effects_l2 = method_effects.loc[method_effects.stage.eq('L2_ml_model')]
figure = effect_grid(method_effects_l2, GENDER_ALL, title='L2 in-processing minus post-processing', scales=scales, facet='window', value='value')
report.figure(figure, method_effects_l2, 'rq7_gender_in_versus_post_processing', "")

In [ ]:
table = effect_table(method_effects, 'female_click_share')
report.table(table, 'rq7_gender_in_versus_post_processing_table', "")

## Coverage and clicking checks for mitigation


In [ ]:
mitigation_trajectories = trajectory.loc[trajectory.study.eq('goodreads_gender_mitigation_a300_e804') & trajectory.stage.eq('L2_ml_model')]
figure = trajectory_grid(mitigation_trajectories, ['weighted_unknown_exposure', 'weighted_ambiguous_exposure', 'click_rate'], title='Gender-mitigation annotation and clicking context', scales=scales, series='condition_label')
report.figure(figure, mitigation_trajectories, 'rq7_gender_mitigation_coverage_click_checks', "")

## Descriptive mean and consistency across available seeds


In [ ]:
from reported_runs.analysis.presentation import cross_seed_table
for metric in ['female_click_share', 'weighted_female_exposure']:
    table = cross_seed_table(results, 'rq7', [metric])
    report.table(table, f'rq7_seed_consistency_{metric}', "")